In [1]:
with open('../data/tiny_shakespeare.txt','r',encoding='utf-8') as f:
    text = f.read()

print(len(text))

1115394


In [2]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [7]:
chars=sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
stoi = {ch:i for i,ch in enumerate(chars)} #dict
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join(itos[i] for i in l)

print(encode('hello world!'))
print(decode(encode('hello world!')))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
hello world!


In [13]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:100])
# print(decode(data[:100].to_numpy()))

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [15]:
# 划分训练集和验证集
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data),len(val_data),len(data),len(train_data)+len(val_data)==len(data))

1003854 111540 1115394 True


In [16]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [20]:
x=train_data[:block_size]
y=train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"{context} -> {target}")


tensor([18]) -> 47
tensor([18, 47]) -> 56
tensor([18, 47, 56]) -> 57
tensor([18, 47, 56, 57]) -> 58
tensor([18, 47, 56, 57, 58]) -> 1
tensor([18, 47, 56, 57, 58,  1]) -> 15
tensor([18, 47, 56, 57, 58,  1, 15]) -> 47
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> 58


In [24]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

print(torch.randint(len(data)-block_size,(batch_size,)))

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data)-block_size,(batch_size,)) #起始位置
    xs = torch.stack([data[i:i+block_size] for i in ix])
    ys = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return xs,ys

tensor([1078327,  453969,   41646,  671252])


In [25]:
xb,yb = get_batch('train')
print(xb.shape,yb.shape)
print(xb)
print(yb)

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46]])
tensor([[43, 60, 43, 52,  1, 63, 43, 39],
        [43, 42,  8,  0, 25, 63,  1, 45],
        [42,  5, 57,  1, 57, 39, 49, 43],
        [57, 58, 63,  6,  1, 58, 46, 47]])


In [28]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"{context} -> {target}")
    print('\n')    

tensor([57]) -> 43
tensor([57, 43]) -> 60
tensor([57, 43, 60]) -> 43
tensor([57, 43, 60, 43]) -> 52
tensor([57, 43, 60, 43, 52]) -> 1
tensor([57, 43, 60, 43, 52,  1]) -> 63
tensor([57, 43, 60, 43, 52,  1, 63]) -> 43
tensor([57, 43, 60, 43, 52,  1, 63, 43]) -> 39


tensor([60]) -> 43
tensor([60, 43]) -> 42
tensor([60, 43, 42]) -> 8
tensor([60, 43, 42,  8]) -> 0
tensor([60, 43, 42,  8,  0]) -> 25
tensor([60, 43, 42,  8,  0, 25]) -> 63
tensor([60, 43, 42,  8,  0, 25, 63]) -> 1
tensor([60, 43, 42,  8,  0, 25, 63,  1]) -> 45


tensor([56]) -> 42
tensor([56, 42]) -> 5
tensor([56, 42,  5]) -> 57
tensor([56, 42,  5, 57]) -> 1
tensor([56, 42,  5, 57,  1]) -> 57
tensor([56, 42,  5, 57,  1, 57]) -> 39
tensor([56, 42,  5, 57,  1, 57, 39]) -> 49
tensor([56, 42,  5, 57,  1, 57, 39, 49]) -> 43


tensor([43]) -> 57
tensor([43, 57]) -> 58
tensor([43, 57, 58]) -> 63
tensor([43, 57, 58, 63]) -> 6
tensor([43, 57, 58, 63,  6]) -> 1
tensor([43, 57, 58, 63,  6,  1]) -> 58
tensor([43, 57, 58, 63,  6,  1, 58])

In [30]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)


class BigramLanguageMode(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets):
        logits = self.token_embedding_table(idx)
        return logits
m = BigramLanguageMode(vocab_size)
out = m(xb,yb)
print(out.shape)




torch.Size([4, 8, 65])
